In [4]:
from __future__ import annotations
import json
from pathlib import Path
from typing import Any, Dict, List
import outlines
import torch
from pydantic import BaseModel, Field
from transformers import AutoModelForCausalLM, AutoTokenizer

TOOLS_PATH = Path("data/ToolVerifier/tools.json")
MODEL_NAME = "Qwen/Qwen3.5-27B"
DEVICE = "cuda:3"
DTYPE = "auto"

TOOL_INDEX = 0
REQUEST_COUNT = 8
MAX_NEW_TOKENS = 2048
TEMPERATURE = 0.8
TOP_P = 0.95


In [5]:
SYSTEM_PROMPT = """You create short, realistic user requests for tool-routing datasets.
Each string must be a natural user request.
Do not mention tool names, JSON, schemas, or implementation details.
Do not number the items."""


class GeneratedQueries(BaseModel):
    requests: List[str] = Field(min_length=REQUEST_COUNT, max_length=REQUEST_COUNT)


In [6]:
def load_tools(path: Path) -> List[Dict[str, Any]]:
    with path.open("r", encoding="utf-8") as handle:
        payload = json.load(handle)
    tools = payload.get("tools")
    if not isinstance(tools, list):
        raise ValueError(f"Expected 'tools' to be a list in {path}")
    return tools


def format_parameters(tool: Dict[str, Any]) -> str:
    parameters = tool.get("parameters", {})
    properties = parameters.get("properties", {})
    required = set(parameters.get("required", []))
    if not properties:
        return "- no parameters"

    lines: List[str] = []
    for name, spec in properties.items():
        if not isinstance(spec, dict):
            continue
        parts = [f"- {name}: {spec.get('type', 'any')}"]
        if spec.get("enum"):
            parts.append(f"choices={spec['enum']}")
        if "default" in spec:
            parts.append(f"default={spec['default']}")
        if name in required:
            parts.append("required")
        lines.append(", ".join(parts))
    return "\n".join(lines) if lines else "- no parameters"


def build_prompt(tool: Dict[str, Any], count: int) -> str:
    name = tool.get("name", "").strip()
    description = tool.get("description", "").strip()
    parameter_text = format_parameters(tool)
    return f"""Create {count} different user requests for this tool.

Tool name: {name}
Tool description: {description}
Parameters:
{parameter_text}

Requirements:
- The response must contain exactly {count} requests.
- Each request should clearly map to this tool.
- Keep the language simple and direct.
- Vary names, locations, dates, numbers, and phrasing.
- Some requests can mention optional parameters when relevant.
- Avoid duplicates.
- Each item in requests should be a standalone natural user request."""


def resolve_dtype(dtype_name: str, device: str) -> torch.dtype:
    if dtype_name == "float16":
        return torch.float16
    if dtype_name == "bfloat16":
        return torch.bfloat16
    if dtype_name == "float32":
        return torch.float32
    if device.startswith("cpu"):
        return torch.float32
    return torch.bfloat16 if torch.cuda.is_available() else torch.float32


def load_generator(model_name: str, device: str, dtype: str):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    if tokenizer.pad_token is None and tokenizer.eos_token is not None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs: Dict[str, Any] = {
        "trust_remote_code": True,
        "dtype": resolve_dtype(dtype, device),
    }
    if device == "auto":
        model_kwargs["device_map"] = "auto"

    hf_model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
    if device != "auto":
        hf_model = hf_model.to(device)
    hf_model.eval()

    structured_model = outlines.from_transformers(hf_model, tokenizer)
    query_generator = outlines.Generator(structured_model, GeneratedQueries)
    return tokenizer, query_generator


@torch.inference_mode()
def generate_queries(
    query_generator,
    tokenizer: AutoTokenizer,
    prompt: str,
    max_new_tokens: int,
    temperature: float,
    top_p: float,
) -> GeneratedQueries:
    full_prompt = f"{SYSTEM_PROMPT}\n\n{prompt}"

    generation_kwargs: Dict[str, Any] = {
        "max_new_tokens": max_new_tokens,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }
    if temperature > 0:
        generation_kwargs.update(
            {
                "do_sample": True,
                "temperature": temperature,
                "top_p": top_p,
            }
        )
    else:
        generation_kwargs["do_sample"] = False

    raw_output = query_generator(full_prompt, **generation_kwargs)
    print("Structured JSON:\n")
    print(raw_output)
    return GeneratedQueries.model_validate_json(raw_output)


In [7]:
tools = load_tools(TOOLS_PATH)
tool = tools[TOOL_INDEX]

print("Selected tool:")
print(json.dumps(tool, indent=2))

prompt = build_prompt(tool, REQUEST_COUNT)
print("\nPrompt:\n")
print(prompt)


Selected tool:
{
  "name": "generate_bank_account_number",
  "description": "Generates a random bank account number for a specified bank.",
  "parameters": {
    "type": "object",
    "properties": {
      "bank_name": {
        "type": "string",
        "default": "Generic Bank"
      },
      "account_type": {
        "type": "string",
        "default": "checking"
      }
    },
    "required": []
  }
}

Prompt:

Create 8 different user requests for this tool.

Tool name: generate_bank_account_number
Tool description: Generates a random bank account number for a specified bank.
Parameters:
- bank_name: string, default=Generic Bank
- account_type: string, default=checking

Requirements:
- The response must contain exactly 8 requests.
- Each request should clearly map to this tool.
- Keep the language simple and direct.
- Vary names, locations, dates, numbers, and phrasing.
- Some requests can mention optional parameters when relevant.
- Avoid duplicates.
- Each item in requests shoul

In [8]:
tokenizer, query_generator = load_generator(MODEL_NAME, DEVICE, DTYPE)

generated_queries = generate_queries(
    query_generator=query_generator,
    tokenizer=tokenizer,
    prompt=prompt,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    top_p=TOP_P,
)

print("\nParsed queries:\n")
for i, query in enumerate(generated_queries.requests, start=1):
    print(f"{i}. {query}")


Fetching 11 files: 100%|██████████| 11/11 [00:00<00:00, 110641.11it/s]
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 851/851 [00:00<00:00, 1554.94it/s, Materializing param=model.norm.weight]                              


Structured JSON:

{ "requests": [ "I need a random account number for Chase.", "Can you generate a savings account number for Wells Fargo?", "Create a checking account number for Bank of America.", "I'm looking for a generic bank account number to test with.", "Give me a random account number for Citibank.", "Make up a savings account number for Capital One.", "I need a checking account number for HSBC.", "Generate a random account number for TD Bank." ] }

Parsed queries:

1. I need a random account number for Chase.
2. Can you generate a savings account number for Wells Fargo?
3. Create a checking account number for Bank of America.
4. I'm looking for a generic bank account number to test with.
5. Give me a random account number for Citibank.
6. Make up a savings account number for Capital One.
7. I need a checking account number for HSBC.
8. Generate a random account number for TD Bank.
